In [0]:
%pip install reverse_geocoder

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from datetime import date, timedelta

dbutils.widgets.text("start_date", "2026-05-11")
dbutils.widgets.text("end_date", "2026-05-12")


start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

silver_adls = "abfss://silver@dataearth.dfs.core.windows.net/"
gold_adls = "abfss://gold@dataearth.dfs.core.windows.net/"

silver_data = f"{silver_adls}earthquake_events_silver/"

In [0]:
from pyspark.sql.functions import col,isnull, when
from pyspark.sql.types import StringType
from datetime import date, timedelta
import reverse_geocoder as rg

In [0]:
df = spark.read.parquet(silver_data).filter(col('time') > start_date)

In [0]:
df = df.limit(100) #added to speed up the processings as during testing it was providing a bottleneck

In [0]:
def get_country_code(lat, lon):
    """
    Retrieve the country code for a given latitude and longitude.

    Parameters:
    lat (float or str): Latitude of the location.
    lon (float or str): Longitude of the location.

    Returns:
    str: Country code of the location, retrieved using the reverse geocoding API.
    """
    try:
        coordinates = (float(lat), float(lon))
        result = rg.search(coordinates)[0].get('cc')
        print(f"Processed coordinates: {coordinates} -> {result}")
        return result
    except Exception as e:
        print(f"Error processing coordinates: {lat}, {lon} -> {str(e)}")
        return None

In [0]:
get_country_code_udf = udf(get_country_code, StringType())

In [0]:
#adding the country_code and city attributes
df_with_location = \
                df.withColumn('country_code', get_country_code_udf(col('latitude'), col('longitude')))


In [0]:
#adding significance classification
df_with_location_sig_class = \
                            df_with_location.\
                                withColumn('sig_class',
                                           when(col('sig')<100, "Low").\
                                               when((col('sig')>=100) & (col('sig')<500), "Moderate").\
                                                otherwise("High")
                                )

In [0]:
df_with_location_sig_class.printSchema()

root
 |-- id: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- elevation: double (nullable = true)
 |-- title: string (nullable = true)
 |-- place_description: string (nullable = true)
 |-- sig: long (nullable = true)
 |-- mag: double (nullable = true)
 |-- magType: string (nullable = true)
 |-- time: timestamp (nullable = true)
 |-- updated: timestamp (nullable = true)
 |-- country_code: string (nullable = true)
 |-- sig_class: string (nullable = false)



In [0]:
print(silver_data)

abfss://silver@dataearth.dfs.core.windows.net/earthquake_events_silver/


In [0]:
gold_outhput_path = f"{gold_adls}earthquake_events_gold/"


In [0]:
df_with_location_sig_class.write.mode("overwrite").parquet(gold_outhput_path)